# Problem Statement : Spam or ham?

## Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

import warnings
warnings.filterwarnings('ignore')

## Load Data

In [2]:
spam = pd.read_csv('spam.csv')
spam.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
spam.tail()

,Category,Message
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...
5571,ham,Rofl. Its true to its name


In [4]:
spam.info()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Category  5572 non-null   str  
 1   Message   5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB


In [5]:
spam.isna().sum()

Category    0
Message     0
dtype: int64

## Feature Extraction : CountVectorizer

In [6]:
train_set = {"The sky is blue.", "The sun is bright."}
test_set = {"The sun in the sky is bright.","we can see the shining sun, the bright sun."}

In [7]:
vectorizer = CountVectorizer()

In [8]:
vectorizer.fit(train_set)
vectorizer.vocabulary_

{'the': 5, 'sky': 3, 'is': 2, 'blue': 0, 'sun': 4, 'bright': 1}

- keys : unique words in the training sentences
- values : indices based on their appearance order when the vocabulary was built

In [9]:
# we can sort the doctionary
voacb_dict = vectorizer.vocabulary_.copy()
dict(sorted(voacb_dict.items(), key=lambda x: x[1]))

{'blue': 0, 'bright': 1, 'is': 2, 'sky': 3, 'sun': 4, 'the': 5}

In [10]:
test_set

{'The sun in the sky is bright.',
 'we can see the shining sun, the bright sun.'}

In [11]:
test_vec = vectorizer.transform(test_set)
print(test_vec)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 8 stored elements and shape (2, 6)>
  Coords	Values
  (0, 1)	1
  (0, 4)	2
  (0, 5)	2
  (1, 1)	1
  (1, 2)	1
  (1, 3)	1
  (1, 4)	1
  (1, 5)	2


- Each entry is represented as `(row_index, column_index)` and `count`
- `row_index`: Indicates which document (from test_set) the entry corresponds to.
- `column_index`: Indicates which word (from the training vocabulary) is being counted.
- `count`: Indicates how many times the word appears in that document.

In [12]:
# convert to array
test_vec.toarray()

array([[0, 1, 0, 0, 2, 2],
       [0, 1, 1, 1, 1, 2]])

In [13]:
# inverse transform
vectorizer.inverse_transform(test_vec)

[array(['bright', 'sun', 'the'], dtype='<U6'),
 array(['bright', 'is', 'sky', 'sun', 'the'], dtype='<U6')]

We only got the token back, not the oreder of words. This is limitation of coutvectorizer.

## Data processing

In [14]:
# Convierting category into binary column
spam['Category']=spam['Category'].apply(lambda x:1 if x=='spam' else 0)
spam.head(5)

,Category,Message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [15]:
X = spam['Message']
y = spam['Category']

In [16]:
X.shape, y.shape

((5572,), (5572,))

In [17]:
X.head()

0    Go until jurong point, crazy.. Available only ...
1                        Ok lar... Joking wif u oni...
2    Free entry in 2 a wkly comp to win FA Cup fina...
3    U dun say so early hor... U c already then say...
4    Nah I don't think he goes to usf, he lives aro...
Name: Message, dtype: str

In [18]:
y.head()

0    0
1    0
2    1
3    0
4    0
Name: Category, dtype: int64

In [19]:
## apply count vectorizer
vect = CountVectorizer(ngram_range=(1,3),min_df=5,max_features=8000)
X_vect = vect.fit_transform(X).toarray()
X_vect[:5]

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(5, 4722))

In [21]:
X_vect.shape

(5572, 4722)

4000+ Features got created by vectorizer.

In [22]:
# split data
X_train, X_test, y_train, y_test = train_test_split(X_vect, y, test_size=0.2, random_state=42)

In [23]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((4457, 4722), (1115, 4722), (4457,), (1115,))

## Build Naive Bayes Model

In [24]:
nb = GaussianNB()
nb.fit(X_train, y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [25]:
# training score
nb.score(X_train, y_train)

0.9479470495849226

In [26]:
# test score
nb.score(X_test, y_test)

0.9452914798206278

## Comparing Naive Bayes with KNN and SVM

In [28]:
# build KNN

%time
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)
knn.score(X_train, y_train), knn.score(X_test, y_test)

CPU times: total: 0 ns
Wall time: 12.9 μs


(0.9360556428090644, 0.9291479820627803)

In [31]:
# build SVM

%time
svm = SVC()
svm.fit(X_train, y_train)
svm.score(X_train, y_train), svm.score(X_test, y_test)

CPU times: total: 0 ns
Wall time: 6.91 μs


(0.9943908458604442, 0.9838565022421525)

In [30]:
# build Naive Bayes

%time
nb = GaussianNB()
nb.fit(X_train, y_train)
nb.score(X_train, y_train), nb.score(X_test, y_test)

CPU times: total: 0 ns
Wall time: 5.48 μs


(0.9479470495849226, 0.9452914798206278)

## Prediction

In [36]:
sample = "You won 25 lakh"
vec = vect.transform([sample]).toarray()
nb.predict(vec)

array([1])